# R09 Flight：EEG、Force 与 Event 对齐

使用两端共同记录的 BCI2000 `SourceTime` 时钟，将 256 Hz 握力数据和任务事件映射到 2000 Hz EEG 采样轴。

## Setup

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from read_bci2000 import BCI2000Dat
from align_flight_eeg import align_force_to_eeg, flight_event_table

data_dir = Path.cwd() / "0807华山grip flight"
eeg = BCI2000Dat(data_dir / "testS001R09.dat.larkcache")
task = BCI2000Dat(data_dir / "testS001R09_1.dat")

## Align

将连续握力插值到 EEG 时间轴，并把任务事件映射为 EEG 样本编号。



In [ ]:
eeg_clock_ms, task_clock_ms, force_at_eeg_rate = align_force_to_eeg(eeg, task)
events = pd.DataFrame(flight_event_table(eeg, task, eeg_clock_ms, task_clock_ms))
events

'''
events:

0 Idle
  ↓ OnTrialBegin
1 Countdown
  ↓ OnFeedbackBegin
2 Playing
  ├─ 到达地图终点 → 4 TrialSuccess
  └─ 碰撞/飞出边界 → 3 Hit → 5 TrialFailure
                         ↓
                    6 InterTrial
                         ↓ 下一个 Trial
                    1 Countdown

Run 结束 → 7 SessionComplete
'''

### 对齐精度检查

In [ ]:
print("最大事件对齐误差 (ms):", events.alignment_error_ms.abs().max())
print("EEG 有效对齐握力点数:", np.isfinite(force_at_eeg_rate).sum(), "/", eeg.samples)
assert events.alignment_error_ms.abs().max() < 0.5

## Plot 20 aligned EEG channels, force, and events

选择前 20 个非空通道，逐通道标准化后纵向堆叠显示。标准化只用于可视化，不修改原始信号。

In [ ]:
valid_indices = [i for i, name in enumerate(eeg.channel_names) if not name.startswith("EMPTY")]
channel_indices = valid_indices[:20]
channel_names = [eeg.channel_names[i] for i in channel_indices]

# 绘图降采样只减少屏幕上的点数，不影响事件对齐或后续 epoch 数据。
plot_step = 4
plot_indices = np.arange(0, eeg.samples, plot_step)
plot_time_s = plot_indices / eeg.sampling_rate
signals_20 = np.asarray(eeg.signals[plot_indices][:, channel_indices], dtype=float)
signals_20 -= np.nanmedian(signals_20, axis=0, keepdims=True)
channel_scale = np.nanstd(signals_20, axis=0, keepdims=True)
channel_scale[channel_scale == 0] = 1
stacked_signals = signals_20 / channel_scale + np.arange(20)[::-1] * 5

fig, (ax_eeg, ax_force) = plt.subplots(
    2, 1, figsize=(17, 12), sharex=True,
    gridspec_kw={"height_ratios": [5, 1]}, constrained_layout=True,
)
for channel_position in range(20):
    ax_eeg.plot(plot_time_s, stacked_signals[:, channel_position], lw=0.35, color="black")
ax_eeg.set_yticks(np.arange(20)[::-1] * 5, labels=channel_names)
ax_eeg.set_ylabel("EEG channel")
ax_eeg.set_title("R09: 20 EEG channels aligned with grip force and flight events")

ax_force.plot(plot_time_s, force_at_eeg_rate[plot_indices], color="tab:orange", lw=0.8)
ax_force.set(xlabel="EEG time (s)", ylabel="GripForce1")

colors = {
    "flight_start": "tab:green",
    "collision_onset": "tab:red",
    "result_onset": "tab:purple",
}
for event_name, color in colors.items():
    event_times = events.loc[events.event == event_name, "eeg_time_s"]
    for event_time in event_times:
        ax_eeg.axvline(event_time, color=color, alpha=0.7, lw=0.9)
        ax_force.axvline(event_time, color=color, alpha=0.7, lw=0.9)
    ax_eeg.plot([], [], color=color, label=event_name)

ax_eeg.legend(loc="upper right", ncol=3)
ax_eeg.grid(axis="x", alpha=0.15)
ax_force.grid(alpha=0.15)
plt.show()

print("Displayed channels:", channel_names)

## Extract event epochs

以每次 `flight_start` 为零点，截取前 1 秒、后 2 秒的 EEG 和握力。

In [ ]:
pre_s, post_s = 1.0, 2.0
relative_time_s = np.arange(-int(pre_s * eeg.sampling_rate), int(post_s * eeg.sampling_rate)) / eeg.sampling_rate
flight_starts = events.query("event == 'flight_start'").eeg_sample.to_numpy(dtype=int)

eeg_epochs = np.stack([
    eeg.signals[start + int(relative_time_s[0] * eeg.sampling_rate):
                start + int(relative_time_s[-1] * eeg.sampling_rate) + 1, :]
    for start in flight_starts
])
force_epochs = np.stack([
    force_at_eeg_rate[start + int(relative_time_s[0] * eeg.sampling_rate):
                      start + int(relative_time_s[-1] * eeg.sampling_rate) + 1]
    for start in flight_starts
])

print("EEG epochs shape (trial, time, channel):", eeg_epochs.shape)
print("Force epochs shape (trial, time):", force_epochs.shape)

In [ ]:
# 1. 数据与事件摘要：Collision 事件只取 0→1 上升沿
from scipy import signal
from IPython.display import display

PALETTE = {
    "blue": "#35618F", "orange": "#D97732", "gold": "#C69C3C",
    "pink": "#A95078", "olive": "#73804A", "grey": "#7A8088",
    "light_grey": "#E6E9ED", "ink": "#252A31",
}
phase_labels = {0:"Idle", 1:"Countdown", 2:"Playing", 3:"Hit",
                4:"Success", 5:"Failure", 6:"InterTrial", 7:"SessionComplete"}
result_labels = {0:"none", 1:"success", 2:"failure"}
state_names = ["GamePhase", "Feedback", "Collision", "CollisionObject",
               "FlightTrialResult", "ResultCode", "BallWorldX", "BallWorldY",
               "BallVelocityY", "GripForceNormalized"]
task_states = {name: task.state(name).astype(np.int64)
               for name in state_names if name in task.state_definitions}
task_time_s = np.arange(task.samples) / task.sampling_rate
eeg_time_s = np.arange(eeg.samples) / eeg.sampling_rate
task_on_eeg_time_s = np.interp(task_clock_ms, eeg_clock_ms, eeg_time_s)

phase = task_states["GamePhase"]
collision_state = task_states["Collision"]
collision_object = task_states["CollisionObject"]
flight_result = task_states["FlightTrialResult"]
collision_onsets_task = np.flatnonzero(
    (collision_state > 0) & np.r_[True, collision_state[:-1] == 0])
collision_events = events.query("event == 'collision_onset'").copy()
assert len(collision_onsets_task) == len(collision_events)

trial_starts_task = np.flatnonzero(
    (phase == 1) & np.r_[True, phase[:-1] != 1])
trial_at_task = np.zeros(task.samples, dtype=int)
trial_rows = []
for trial_number, start in enumerate(trial_starts_task, 1):
    end = trial_starts_task[trial_number] if trial_number < len(trial_starts_task) else task.samples
    trial_at_task[start:end] = trial_number
    trial_collision_onsets = collision_onsets_task[
        (collision_onsets_task >= start) & (collision_onsets_task < end)]
    outcome_code = int(flight_result[start:end].max())
    trial_rows.append({
        "trial": trial_number, "start_s": start / task.sampling_rate,
        "duration_s": (end - start) / task.sampling_rate,
        "outcome": result_labels[outcome_code], "outcome_code": outcome_code,
        "collision_onsets": len(trial_collision_onsets),
    })
trial_summary = pd.DataFrame(trial_rows)

event_audit = events.copy()
event_audit["label"] = [
    phase_labels.get(int(row.code), f"phase_{int(row.code)}")
    if row.event in {"trial_start", "flight_start"}
    else f"object_{int(collision_object[int(row.task_sample)])}"
    if row.event == "collision_onset"
    else result_labels.get(int(row.code), f"result_{int(row.code)}")
    for row in event_audit.itertuples()
]
recording_summary = pd.Series({
    "EEG channels": eeg.source_channels,
    "EEG samples": eeg.samples,
    "EEG sampling rate (Hz)": eeg.sampling_rate,
    "EEG duration (s)": eeg.samples / eeg.sampling_rate,
    "Task samples": task.samples,
    "Task duration (s)": task.samples / task.sampling_rate,
    "Trials": len(trial_summary),
    "Success trials": int((trial_summary.outcome_code == 1).sum()),
    "Failure trials": int((trial_summary.outcome_code == 2).sum()),
    "Collision onsets": len(collision_onsets_task),
}, name="value")
display(recording_summary.to_frame())
display(trial_summary.drop(columns="outcome_code"))
display(event_audit[["trial", "event", "label", "task_time_s",
                     "eeg_time_s", "alignment_error_ms"]]) 
''' 
一个 EEG 采样间隔 = 1 / 2000 s = 0.5 ms
alignment_error_ms符号含义:
正数：选中的 EEG 采样点比理想映射时刻稍晚
负数：选中的 EEG 采样点比理想映射时刻稍早
'''

In [ ]:
# 2. 全程信号与标签审计：共享时间轴，Collision 虚线、Result 点线
valid_indices = [i for i, name in enumerate(eeg.channel_names)
                 if not name.startswith("EMPTY")]
overview_channels = valid_indices[:32]
overview_names = [eeg.channel_names[i] for i in overview_channels]
overview_step = max(1, int(eeg.sampling_rate // 100))
overview_indices = np.arange(0, eeg.samples, overview_step)
overview_eeg = np.asarray(
    eeg.signals[overview_indices][:, overview_channels], dtype=np.float32)
overview_center = np.median(overview_eeg, axis=0, keepdims=True)
overview_scale = np.median(
    np.abs(overview_eeg - overview_center), axis=0, keepdims=True) * 1.4826
overview_scale[overview_scale == 0] = 1
overview_z = np.clip((overview_eeg - overview_center) / overview_scale, -4, 4)

task_plot_indices = np.arange(0, task.samples, 8)
ball_y = task_states["BallWorldY"].astype(float)
ball_velocity_y = task_states["BallVelocityY"].astype(float) - 32768.0
audit_names = ["Feedback", "Collision", "CollisionObject",
               "FlightTrialResult", "ResultCode"]
height_ratios = [5, 1.2, 1.2] + [0.22] * len(audit_names)
fig, axes = plt.subplots(
    3 + len(audit_names), 1, figsize=(18, 14), sharex=True,
    gridspec_kw={"height_ratios": height_ratios},
    constrained_layout=True)

image = axes[0].imshow(
    overview_z.T, aspect="auto", origin="lower", interpolation="nearest",
    extent=[eeg_time_s[overview_indices[0]], eeg_time_s[overview_indices[-1]],
            0, len(overview_channels)],
    cmap="RdBu_r", vmin=-3, vmax=3)
axes[0].set(ylabel="EEG channel index",
            title="R09 signals and event labels over time")
fig.colorbar(image, ax=axes[0], label="Robust z-score", pad=0.01)

axes[1].plot(eeg_time_s[overview_indices],
             force_at_eeg_rate[overview_indices],
             color=PALETTE["orange"], lw=0.8)
axes[1].set_ylabel("Grip force")
axes[2].plot(task_on_eeg_time_s[task_plot_indices],
             ball_y[task_plot_indices], color=PALETTE["blue"], lw=0.9)
velocity_axis = axes[2].twinx()
velocity_axis.plot(task_on_eeg_time_s[task_plot_indices],
                   ball_velocity_y[task_plot_indices],
                   color=PALETTE["gold"], lw=0.7)
axes[2].set_ylabel("Ball Y")
velocity_axis.set_ylabel("Velocity Y")

audit_axes = axes[3:]
for ax, name in zip(audit_axes, audit_names):
    ax.step(task_on_eeg_time_s, task_states[name], where="post",
            color=PALETTE["blue"], lw=0.8)
    ax.set_ylabel(name, rotation=0, ha="right", va="center", fontsize=8)
    ax.set_yticks([])

for ax in axes:
    for collision_time in collision_events.eeg_time_s:
        ax.axvline(collision_time, color=PALETTE["orange"], ls="--", lw=1.3)
    for result_time in events.query("event == 'result_onset'").eeg_time_s:
        ax.axvline(result_time, color=PALETTE["pink"], ls=":", lw=1.0)
    ax.grid(axis="x", color=PALETTE["light_grey"], lw=0.6)
axes[-1].set_xlabel("EEG-aligned time (s)")
plt.show()

label_quality = pd.Series({
    "Collision positive samples": int((collision_state > 0).sum()),
    "Collision onset events": len(collision_onsets_task),
    "Collision outside Playing samples":
        int(((collision_state > 0) & (phase != 2)).sum()),
    "CollisionObject without Collision samples":
        int(((collision_object > 0) & (collision_state == 0)).sum()),
    "Collision without object samples":
        int(((collision_state > 0) & (collision_object == 0)).sum()),
}, name="count")
display(label_quality.to_frame())

In [ ]:
# 4. Trial 对比：以 flight_start 为零点
flight_starts_task = np.flatnonzero(
    (phase == 2) & np.r_[True, phase[:-1] != 2])
fig, axes = plt.subplots(3, 1, figsize=(16, 11),
                         sharex=True, constrained_layout=True)
for trial_number, flight_start in enumerate(flight_starts_task, 1):
    trial_start = trial_starts_task[trial_number - 1]
    trial_end = (trial_starts_task[trial_number]
                 if trial_number < len(trial_starts_task) else task.samples)
    outcome_code = int(flight_result[trial_start:trial_end].max())
    outcome = result_labels[outcome_code]
    color = PALETTE["orange"] if outcome_code == 2 else PALETTE["blue"]
    line_style = "--" if outcome_code == 2 else "-"
    indices = np.arange(trial_start, trial_end, 4)
    relative_time = (indices - flight_start) / task.sampling_rate
    label = f"Trial {trial_number}: {outcome}"
    axes[0].plot(relative_time,
                 np.asarray(task.signals[indices, 0], dtype=float),
                 color=color, ls=line_style, lw=1.2, alpha=0.9, label=label)
    axes[1].plot(relative_time, ball_y[indices],
                 color=color, ls=line_style, lw=1.1, alpha=0.9)
    axes[2].plot(relative_time, ball_velocity_y[indices],
                 color=color, ls=line_style, lw=1.0, alpha=0.9)
for ax in axes:
    ax.axvline(0, color=PALETTE["ink"], lw=1.0)
    ax.grid(color=PALETTE["light_grey"], lw=0.7)
axes[0].set(title="Trial-aligned behavior", ylabel="GripForce1")
axes[1].set_ylabel("BallWorldY")
axes[2].set(xlabel="Time from flight_start (s)", ylabel="BallVelocityY")
axes[0].legend(ncol=2, frameon=False)
plt.show()

In [ ]:
# 5. Collision onset 放大：-2 到 +3 秒，n=1，仅作描述
if len(collision_events) == 0:
    print("No collision_onset found.")
else:
    collision_row = collision_events.iloc[0]
    collision_eeg_sample = int(collision_row.eeg_sample)
    collision_task_sample = int(collision_row.task_sample)
    pre_collision_s, post_collision_s = 2.0, 3.0
    eeg_offsets = np.arange(-int(pre_collision_s * eeg.sampling_rate),
                            int(post_collision_s * eeg.sampling_rate))
    eeg_window_indices = collision_eeg_sample + eeg_offsets
    valid = (eeg_window_indices >= 0) & (eeg_window_indices < eeg.samples)
    eeg_window_indices = eeg_window_indices[valid]
    collision_relative_eeg_s = eeg_offsets[valid] / eeg.sampling_rate

    collision_channels = overview_channels[:12]
    collision_names = [eeg.channel_names[i] for i in collision_channels]
    collision_eeg = np.asarray(
        eeg.signals[eeg_window_indices][:, collision_channels],
        dtype=np.float32)
    collision_eeg -= np.median(collision_eeg, axis=0, keepdims=True)
    scale = np.std(collision_eeg, axis=0, keepdims=True)
    scale[scale == 0] = 1
    collision_stacked = (collision_eeg / scale
                         + np.arange(len(collision_channels))[::-1] * 5)

    task_offsets = np.arange(-int(pre_collision_s * task.sampling_rate),
                             int(post_collision_s * task.sampling_rate))
    task_window_indices = collision_task_sample + task_offsets
    valid_task = ((task_window_indices >= 0)
                  & (task_window_indices < task.samples))
    task_window_indices = task_window_indices[valid_task]
    collision_relative_task_s = task_offsets[valid_task] / task.sampling_rate

    fig, axes = plt.subplots(
        3, 1, figsize=(16, 12), sharex=True,
        gridspec_kw={"height_ratios":[5, 1.5, 1.8]},
        constrained_layout=True)
    for pos in range(len(collision_channels)):
        axes[0].plot(collision_relative_eeg_s, collision_stacked[:, pos],
                     color=PALETTE["ink"], lw=0.45)
    axes[0].set_yticks(np.arange(len(collision_channels))[::-1] * 5,
                       labels=collision_names)
    axes[0].set(ylabel="EEG channel",
                title="Single collision_onset window (n=1; descriptive only)")
    axes[1].plot(collision_relative_eeg_s,
                 force_at_eeg_rate[eeg_window_indices],
                 color=PALETTE["orange"], lw=1.0)
    axes[1].set_ylabel("Grip force")
    axes[2].step(collision_relative_task_s, phase[task_window_indices],
                 where="post", color=PALETTE["blue"], lw=1.2,
                 label="GamePhase")
    axes[2].step(collision_relative_task_s,
                 collision_state[task_window_indices] + 7,
                 where="post", color=PALETTE["orange"], lw=1.2,
                 label="Collision + 7")
    axes[2].step(collision_relative_task_s,
                 flight_result[task_window_indices] + 9,
                 where="post", color=PALETTE["pink"], lw=1.2,
                 label="FlightTrialResult + 9")
    axes[2].set(xlabel="Time from collision_onset (s)",
                ylabel="State code")
    axes[2].legend(ncol=3, frameon=False)
    for ax in axes:
        ax.axvline(0, color=PALETTE["orange"], ls="--", lw=1.5)
        ax.grid(color=PALETTE["light_grey"], lw=0.7)
    plt.show()
    display(event_audit.query("event in ['collision_onset', 'result_onset']"))

In [ ]:
# 6. EEG 数据质量与 0–300 Hz PSD
quality_channels = valid_indices
quality_names = [eeg.channel_names[i] for i in quality_channels]
quality_step = max(1, int(eeg.sampling_rate // 100))
quality_data = np.asarray(
    eeg.signals[::quality_step][:, quality_channels], dtype=np.float32)
channel_std = np.std(quality_data, axis=0)
channel_range = (np.percentile(quality_data, 99, axis=0)
                 - np.percentile(quality_data, 1, axis=0))
channel_zero_fraction = np.mean(quality_data == 0, axis=0)

psd_samples = min(int(20 * eeg.sampling_rate), eeg.samples)
psd_data = np.asarray(
    eeg.signals[:psd_samples][:, quality_channels], dtype=np.float32)
frequencies, psd = signal.welch(
    psd_data, fs=eeg.sampling_rate, nperseg=min(4096, psd_samples),
    axis=0, detrend="constant")
frequency_mask = (frequencies >= 0) & (frequencies <= 300)
psd_db = 10 * np.log10(psd[frequency_mask].T + np.finfo(float).tiny)

fig = plt.figure(figsize=(18, 18), constrained_layout=True)
quality_grid = fig.add_gridspec(
    4, 2, height_ratios=[1.2, 1.2, 4, 4], width_ratios=[1, 1])
axes = [
    fig.add_subplot(quality_grid[0, :]),
    fig.add_subplot(quality_grid[1, :]),
    fig.add_subplot(quality_grid[2, 0]),
    fig.add_subplot(quality_grid[3, :]),
]
channel_indices = np.arange(len(quality_channels))
axes[0].plot(channel_indices, channel_std,
             color=PALETTE["blue"], lw=1.1)
axes[0].set(title="EEG channel standard deviation",
            ylabel="SD (raw ADC units)",
            xlim=(-1, len(quality_channels)))
axes[1].plot(channel_indices, channel_range,
             color=PALETTE["orange"], lw=1.1)
axes[1].set(title="EEG channel robust amplitude range",
            ylabel="P99–P1 (raw ADC units)",
            xlim=(-1, len(quality_channels)))
axes[2].barh(channel_indices, channel_zero_fraction,
             color=PALETTE["grey"], height=0.9)
axes[2].set(xlabel="Zero fraction", ylabel="Channel index",
            xlim=(0, 1),
            ylim=(-1, len(quality_channels)))
psd_image = axes[3].imshow(
    psd_db, aspect="auto", origin="lower", interpolation="nearest",
    extent=[frequencies[frequency_mask][0], frequencies[frequency_mask][-1],
            0, len(quality_channels)], cmap="magma")
axes[3].axvline(50, color="white", ls="--", lw=0.9)
axes[3].set(xlabel="Frequency (Hz)", ylabel="Channel index")
fig.colorbar(psd_image, ax=axes[3],
             label="PSD (dB, raw units²/Hz)", pad=0.01)
for ax in axes[:3]:
    ax.grid(color=PALETTE["light_grey"], lw=0.7)
plt.show()
quality_summary = pd.DataFrame({
    "channel": quality_names, "std": channel_std,
    "p99_p1_range": channel_range,
    "zero_fraction": channel_zero_fraction,
}).sort_values("std", ascending=False)
display(quality_summary.head(15))
print("Flat channels:", int((channel_std == 0).sum()))

In [ ]:
# 7. Flight-start 事件相关 EEG：个体 Trial、GFP 热图和平均通道响应
epoch_channels = overview_channels[:20]
epoch_pre_s, epoch_post_s = 1.0, 2.0
epoch_offsets = np.arange(-int(epoch_pre_s * eeg.sampling_rate),
                          int(epoch_post_s * eeg.sampling_rate))
epoch_time_s = epoch_offsets / eeg.sampling_rate
flight_starts_eeg = events.query(
    "event == 'flight_start'").eeg_sample.to_numpy(dtype=int)
valid_flight_starts = flight_starts_eeg[
    (flight_starts_eeg + epoch_offsets[0] >= 0)
    & (flight_starts_eeg + epoch_offsets[-1] < eeg.samples)]
flight_epochs = np.stack([
    np.asarray(eeg.signals[start + epoch_offsets][:, epoch_channels],
               dtype=np.float32)
    for start in valid_flight_starts])
baseline_mask = epoch_time_s < 0
baseline_center = np.mean(
    flight_epochs[:, baseline_mask, :], axis=1, keepdims=True)
baseline_scale = np.std(
    flight_epochs[:, baseline_mask, :], axis=1, keepdims=True)
baseline_scale[baseline_scale == 0] = 1
flight_epochs_z = (flight_epochs - baseline_center) / baseline_scale
flight_gfp = np.sqrt(np.mean(flight_epochs_z ** 2, axis=2))
mean_channel_response = np.mean(flight_epochs_z, axis=0).T

fig, axes = plt.subplots(
    3, 1, figsize=(16, 13),
    gridspec_kw={"height_ratios":[1.6, 2.0, 4.0]},
    constrained_layout=True)
for trial_index, trace in enumerate(flight_gfp, 1):
    axes[0].plot(epoch_time_s, trace, lw=1.0, alpha=0.5,
                 label=f"Trial {trial_index}")
axes[0].set(title="Flight-start EEG response",
            ylabel="GFP (baseline z)")
axes[0].legend(ncol=4, frameon=False)
gfp_image = axes[1].imshow(
    flight_gfp, aspect="auto", origin="lower",
    extent=[epoch_time_s[0], epoch_time_s[-1],
            0.5, len(flight_gfp) + 0.5], cmap="viridis")
axes[1].set(ylabel="Trial",
            yticks=np.arange(1, len(flight_gfp) + 1))
fig.colorbar(gfp_image, ax=axes[1], label="GFP", pad=0.01)
response_image = axes[2].imshow(
    np.clip(mean_channel_response, -3, 3), aspect="auto",
    origin="lower",
    extent=[epoch_time_s[0], epoch_time_s[-1],
            0, len(epoch_channels)],
    cmap="RdBu_r", vmin=-3, vmax=3)
axes[2].set(xlabel="Time from flight_start (s)",
            ylabel="Channel index")
fig.colorbar(response_image, ax=axes[2],
             label="Mean baseline z", pad=0.01)
for ax in axes:
    ax.axvline(0, color=PALETTE["ink"], lw=1.0)
plt.show()

In [ ]:
# 8. Trial onset 时频：以 trial_start（Countdown 开始）锁时并跨 Trial 平均
trial_onset_events = events.query("event == 'trial_start'").copy()
if len(trial_onset_events) == 0:
    print("No trial_start found.")
else:
    tf_channels = overview_channels[:20]
    trial_onset_centers = trial_onset_events.eeg_sample.to_numpy(dtype=int)
    tf_offsets = np.arange(-int(1 * eeg.sampling_rate),
                           int(3 * eeg.sampling_rate))
    tf_time_s = tf_offsets / eeg.sampling_rate
    valid_trial_centers = trial_onset_centers[
        (trial_onset_centers + tf_offsets[0] >= 0)
        & (trial_onset_centers + tf_offsets[-1] < eeg.samples)]
    tf_continuous = np.asarray(
        eeg.signals[:, tf_channels], dtype=np.float32)
    tf_continuous -= np.mean(tf_continuous, axis=0, keepdims=True)
    frequency_bands = {
        "Theta 4–8 Hz":(4, 8), "Alpha 8–13 Hz":(8, 13),
        "Beta 13–30 Hz":(13, 30), "Low gamma 30–45 Hz":(30, 45),
        "High gamma 70–150 Hz":(70, 150)}
    tf_baseline = (tf_time_s >= -1) & (tf_time_s <= -0.2)
    band_power_db = {}
    for band_name, (low_hz, high_hz) in frequency_bands.items():
        band_filter = signal.butter(
            4, [low_hz, high_hz], btype="bandpass",
            fs=eeg.sampling_rate, output="sos")
        filtered = signal.sosfiltfilt(
            band_filter, tf_continuous, axis=0)
        continuous_power = (
            np.abs(signal.hilbert(filtered, axis=0)) ** 2)
        power = np.stack([
            continuous_power[center + tf_offsets]
            for center in valid_trial_centers])
        baseline_power = np.mean(
            power[:, tf_baseline, :], axis=1, keepdims=True)
        baseline_power[baseline_power == 0] = 1
        trial_power_db = 10 * np.log10(
            power / baseline_power + np.finfo(float).tiny)
        band_power_db[band_name] = np.mean(trial_power_db, axis=0).T

    fig, axes = plt.subplots(
        len(band_power_db), 1,
        figsize=(16, 3.2 * len(band_power_db)), sharex=True,
        constrained_layout=True)
    for ax, (band_name, power_db) in zip(axes, band_power_db.items()):
        image = ax.imshow(
            np.clip(power_db, -8, 8), aspect="auto", origin="lower",
            extent=[tf_time_s[0], tf_time_s[-1], 0, len(tf_channels)],
            cmap="RdBu_r", vmin=-6, vmax=6)
        ax.axvline(0, color=PALETTE["ink"], ls="--", lw=1.2)
        ax.set(ylabel="Channel", title=band_name, yticks=[])
        fig.colorbar(image, ax=ax, label="Power change (dB)", pad=0.01)
    axes[-1].set_xlabel("Time from trial_start onset (s)")
    fig.suptitle(
        f"Trial-onset band power (n={len(valid_trial_centers)} trials)")
    plt.show()

In [ ]:
# 9. EEG–握力关系：五个频段分别绘制 hexbin，并汇总滞后相关
relationship_channels = overview_channels[:20]
relationship_data = np.asarray(
    eeg.signals[:, relationship_channels], dtype=np.float32)
relationship_data -= np.mean(relationship_data, axis=0, keepdims=True)
relationship_bands = {
    "0–4 Hz": (0, 4),
    "7–20 Hz": (7, 20),
    "70–115 Hz": (70, 115),
    "130–200 Hz": (130, 200),
    "200–300 Hz": (200, 300),
}
band_colors = {
    "0–4 Hz": PALETTE["blue"],
    "7–20 Hz": PALETTE["orange"],
    "70–115 Hz": PALETTE["gold"],
    "130–200 Hz": PALETTE["pink"],
    "200–300 Hz": PALETTE["olive"],
}
power_smoother = signal.butter(
    4, 2, btype="lowpass", fs=eeg.sampling_rate, output="sos")
relationship_step = max(1, int(eeg.sampling_rate // 50))
relationship_rate = eeg.sampling_rate / relationship_step
force_50hz = force_at_eeg_rate[::relationship_step]
relationship_features_50hz = {}
for band_name, (low_hz, high_hz) in relationship_bands.items():
    if low_hz == 0:
        band_filter = signal.butter(
            4, high_hz, btype="lowpass",
            fs=eeg.sampling_rate, output="sos")
    else:
        band_filter = signal.butter(
            4, [low_hz, high_hz], btype="bandpass",
            fs=eeg.sampling_rate, output="sos")
    band_signal = signal.sosfiltfilt(
        band_filter, relationship_data, axis=0)
    band_log_power = np.log1p(np.mean(band_signal ** 2, axis=1))
    band_feature = signal.sosfiltfilt(power_smoother, band_log_power)
    relationship_features_50hz[band_name] = (
        band_feature[::relationship_step])

lag_seconds = np.arange(-1, 1.0001, 0.02)
lag_samples = np.rint(lag_seconds * relationship_rate).astype(int)
band_lag_correlations = {}
for band_name, feature_50hz in relationship_features_50hz.items():
    lag_correlations = []
    for lag in lag_samples:
        if lag > 0:
            x_values, y_values = feature_50hz[:-lag], force_50hz[lag:]
        elif lag < 0:
            x_values, y_values = feature_50hz[-lag:], force_50hz[:lag]
        else:
            x_values, y_values = feature_50hz, force_50hz
        finite_lag = np.isfinite(x_values) & np.isfinite(y_values)
        lag_correlations.append(
            np.corrcoef(x_values[finite_lag], y_values[finite_lag])[0, 1]
            if finite_lag.sum() >= 3 else np.nan)
    band_lag_correlations[band_name] = np.asarray(lag_correlations)

fig, axes = plt.subplots(2, 3, figsize=(18, 11),
                         constrained_layout=True)
axes = axes.ravel()
finite_force = force_50hz[np.isfinite(force_50hz)]
force_limits = (finite_force.min(), finite_force.max())
for ax, (band_name, feature_50hz) in zip(
        axes[:5], relationship_features_50hz.items()):
    finite = np.isfinite(feature_50hz) & np.isfinite(force_50hz)
    concurrent_r = np.corrcoef(
        feature_50hz[finite], force_50hz[finite])[0, 1]
    hexbin = ax.hexbin(
        feature_50hz[finite], force_50hz[finite],
        gridsize=45, mincnt=1, cmap="Blues")
    ax.set(title=f"{band_name}: concurrent r={concurrent_r:.3f}",
           xlabel="Smoothed log band power", ylabel="Grip force",
           ylim=force_limits)
    fig.colorbar(hexbin, ax=ax, label="Sample count", pad=0.01)

lag_axis = axes[5]
for band_name, lag_correlations in band_lag_correlations.items():
    lag_axis.plot(
        lag_seconds, lag_correlations, label=band_name,
        color=band_colors[band_name], lw=1.4)
    best_index = int(np.nanargmax(np.abs(lag_correlations)))
    lag_axis.scatter(
        lag_seconds[best_index], lag_correlations[best_index],
        color=band_colors[band_name], s=22, zorder=3)
    print(
        f"{band_name}: strongest |r|={lag_correlations[best_index]:.3f} "
        f"at lag={lag_seconds[best_index]:+.2f} s")
lag_axis.axhline(0, color=PALETTE["grey"], lw=0.8)
lag_axis.axvline(0, color=PALETTE["grey"], lw=0.8)
lag_axis.set(
    title="Lagged correlation by frequency band",
    xlabel="Lag (s); positive = EEG leads force",
    ylabel="Pearson correlation")
lag_axis.legend(frameon=False, fontsize=8)
lag_axis.grid(color=PALETTE["light_grey"], lw=0.7)
fig.suptitle(
    "Multi-band EEG power vs grip force (whole recording; exploratory)")
plt.show()
print("Exploratory only: adjacent samples are autocorrelated; "
      "lag maxima are selected after scanning multiple lags.")
print("High-frequency bands include possible 50 Hz line-noise harmonics.")